# Lesson 2 — PyTorch Broadcasting

## 学习目标

完成这一节后，应能够：

1. 判断两个 Tensor 是否可以 broadcasting；
2. 推导 broadcasting 后的 shape；
3. 理解为什么 size=1 的维度很特殊；
4. 理解 reduction 和 `keepdim=True`；
5. 理解 `unsqueeze()`；
6. 看懂 Transformer 中 Bias、RMSNorm 和 Attention Mask 的 broadcasting。


## 1. Broadcasting

Broadcasting 允许不同 shape 的 Tensor 在满足一定条件时进行逐元素运算。

例如：

$$
x.shape=(2,3,4)
$$

而：

$$
bias.shape=(4,)
$$

仍然可以直接计算：

`x + bias`

PyTorch 可以把 `(4,)` 在逻辑上理解为：

$$
(1,1,4)
$$

于是：

$$
(2,3,4)
+
(1,1,4)
\rightarrow
(2,3,4)
$$

也就是说，同一份 bias 被应用到了所有 batch 和所有 token。


In [1]:
import torch

x = torch.randn(2, 3, 4)
bias = torch.randn(4)

y = x + bias

print("x    :", x.shape)
print("bias :", bias.shape)
print("y    :", y.shape)

x    : torch.Size([2, 3, 4])
bias : torch.Size([4])
y    : torch.Size([2, 3, 4])


## 2. Broadcasting Rule

判断两个 Tensor 是否可以 broadcasting：

> 从最右边的维度开始比较。

对应的两个维度满足以下任意条件即可：

1. 两个维度大小相同；
2. 其中一个维度大小为 1。

如果一个 Tensor 的维度更少，可以在左边把缺失维度理解成 1。


### Example 1

$$
A=(2,3,4)
$$

$$
B=(4,)
$$

右对齐：

$$
A=(2,3,4)
$$

$$
B=(1,1,4)
$$

逐维比较：

- 4 vs 4：兼容
- 3 vs 1：兼容
- 2 vs 1：兼容

所以：

$$
A+B
$$

结果 shape：

$$
(2,3,4)
$$


### Example 2

$$
A=(2,3,4)
$$

$$
B=(3,1)
$$

右对齐：

$$
A=(2,3,4)
$$

$$
B=(1,3,1)
$$

比较：

- 4 vs 1：兼容
- 3 vs 3：兼容
- 2 vs 1：兼容

最终：

$$
(2,3,4)
$$


## 3. Broadcasting Failure

例如：

$$
A=(2,3,4)
$$

$$
B=(3,)
$$

右对齐：

$$
A=(2,3,4)
$$

$$
B=(1,1,3)
$$

最后一个维度：

$$
4 \neq 3
$$

并且：

$$
4 \neq 1,\quad 3 \neq 1
$$

因此无法 broadcasting。


In [2]:
x = torch.randn(2, 3, 4)
y = torch.randn(3)

try:
    z = x + y
except RuntimeError as error:
    print(error)


The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 2


## 4. 为什么维度 1 可以扩展？

假设：

$$
x.shape=(2,3,4)
$$

$$
scale.shape=(1,1,4)
$$

Broadcasting 可以理解为：

第 0 维：

$$
1 \rightarrow 2
$$

第 1 维：

$$
1 \rightarrow 3
$$

第 2 维：

$$
4 \rightarrow 4
$$

所以逻辑上：

$$
(1,1,4)
\rightarrow
(2,3,4)
$$

但是 PyTorch 通常不需要真的复制所有数据。

因此 broadcasting 可以同时做到：

- 代码简洁；
- 避免不必要的数据复制。


## 5. Linear Layer 中的 Broadcasting

Linear Layer：

$$
Y=XW+b
$$

假设：

$$
X.shape=(B,T,D)
$$

$$
W.shape=(D,H)
$$

矩阵乘法后：

$$
XW.shape=(B,T,H)
$$

bias 只需要：

$$
b.shape=(H,)
$$

Broadcasting 自动把它理解成：

$$
(1,1,H)
$$

因此：

$$
(B,T,H)
+
(1,1,H)
\rightarrow
(B,T,H)
$$

不需要为每个 batch、每个 token 单独保存 bias。


In [3]:
B = 2
T = 3
D = 4
H = 6

x = torch.randn(B, T, D)
weight = torch.randn(D, H)
bias = torch.randn(H)

y = x @ weight + bias

print("x     :", x.shape)
print("weight:", weight.shape)
print("bias  :", bias.shape)
print("y     :", y.shape)


x     : torch.Size([2, 3, 4])
weight: torch.Size([4, 6])
bias  : torch.Size([6])
y     : torch.Size([2, 3, 6])


## 6. Reduction

Reduction 指的是沿某个维度进行聚合，例如：

- mean
- sum
- max
- min

例如：

$$
x.shape=(2,3,4)
$$

执行：

`x.mean(dim=-1)`

表示沿最后一个维度求平均。

于是：

$$
(2,3,4)
\rightarrow
(2,3)
$$

最后一个维度被 reduction 掉。


In [4]:
x = torch.randn(2, 3, 4)

mean = x.mean(dim=1)

print("x   :", x.shape)
print("mean:", mean.shape)


x   : torch.Size([2, 3, 4])
mean: torch.Size([2, 4])


## 7. keepdim=True

如果写：

`x.mean(dim=-1, keepdim=True)`

被 reduction 的维度不会完全消失，而是变成长度为 1 的维度。

因此：

$$
(2,3,4)
\rightarrow
(2,3,1)
$$

为什么要保留这个 1？

因为：

$$
(2,3,4)
$$

和：

$$
(2,3,1)
$$

可以直接 broadcasting。

这也是 normalization 中经常出现 `keepdim=True` 的重要原因。


In [5]:
x = torch.randn(2, 3, 4)

a = x.mean(dim=-1)
b = x.mean(dim=-1, keepdim=True)

print("without keepdim:", a.shape)
print("with keepdim   :", b.shape)


without keepdim: torch.Size([2, 3])
with keepdim   : torch.Size([2, 3, 1])


## 8. unsqueeze

如果已经得到：

$$
mean.shape=(2,3)
$$

可以使用：

`mean.unsqueeze(-1)`

人为增加一个新的最后维度：

$$
(2,3)
\rightarrow
(2,3,1)
$$

因此在 shape 层面：

`x.mean(dim=-1, keepdim=True)`

和：

`x.mean(dim=-1).unsqueeze(-1)`

得到相同的 shape。


In [6]:
x = torch.randn(2, 3, 4)

a = x.mean(dim=-1, keepdim=True)
b = x.mean(dim=-1).unsqueeze(-1)

print("a:", a.shape)
print("b:", b.shape)

print("same values:", torch.allclose(a, b))


a: torch.Size([2, 3, 1])
b: torch.Size([2, 3, 1])
same values: True


## 9. RMSNorm 中的 Broadcasting

RMSNorm 中需要计算：

$$
\operatorname{RMS}(x)
=
\sqrt{
\frac{1}{D}
\sum_{i=1}^{D}x_i^2+\epsilon
}
$$

如果：

$$
x.shape=(B,T,D)
$$

执行：

`x.pow(2).mean(dim=-1, keepdim=True)`

shape 变化：

$$
(B,T,D)
\rightarrow
(B,T,1)
$$

因此：

`x / rms`

对应：

$$
(B,T,D)
/
(B,T,1)
$$

最后一个维度：

$$
D \text{ vs } 1
$$

可以 broadcasting。

最终输出仍然是：

$$
(B,T,D)
$$


In [7]:
B = 2
T = 3
D = 4

x = torch.randn(B, T, D)

eps = 1e-6

rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)

normalized = x / rms

print("x         :", x.shape)
print("rms       :", rms.shape)
print("normalized:", normalized.shape)


x         : torch.Size([2, 3, 4])
rms       : torch.Size([2, 3, 1])
normalized: torch.Size([2, 3, 4])


## 10. Attention Mask 中的 Broadcasting

Self-Attention 后面会得到 Attention Scores：

$$
scores.shape=(B,H,T,T)
$$

例如：

$$
(2,8,128,128)
$$

但 causal mask 通常只需要：

$$
mask.shape=(T,T)
$$

例如：

$$
(128,128)
$$

右对齐以后相当于：

$$
scores=(B,H,T,T)
$$

$$
mask=(1,1,T,T)
$$

所以：

- 第一个 1 broadcast 到 B；
- 第二个 1 broadcast 到 H。

因此只需要保存一份 causal mask。

不需要为每个 batch 和每个 attention head 分别复制 mask。


In [8]:
B = 2
H = 4
T = 8

scores = torch.randn(B, H, T, T)

mask = torch.triu(
    torch.ones(T, T, dtype=torch.bool),
    diagonal=1,
)

masked_scores = scores.masked_fill(
    mask,
    float("-inf"),
)

print("scores:", scores.shape)
print("mask  :", mask.shape)
print("result:", masked_scores.shape)


scores: torch.Size([2, 4, 8, 8])
mask  : torch.Size([8, 8])
result: torch.Size([2, 4, 8, 8])


## 11. Broadcasting 判断方法

以后看到两个不同 shape 的 Tensor，可以固定使用下面的方法。

### Step 1

写出两个 shape。

### Step 2

右对齐。

### Step 3

从最右边开始逐维比较。

每一维必须满足：

$$
a=b
$$

或者：

$$
a=1
$$

或者：

$$
b=1
$$

### Step 4

最终对应维度取广播后的较大值。

例如：

$$
A=(2,8,128,64)
$$

$$
B=(128,1)
$$

右对齐：

$$
A=(2,8,128,64)
$$

$$
B=(1,1,128,1)
$$

逐维比较：

- 64 vs 1 -> 64
- 128 vs 128 -> 128
- 8 vs 1 -> 8
- 2 vs 1 -> 2

最终：

$$
(2,8,128,64)
$$


## 本节总结

### Rule 1

Broadcasting 从最右边开始比较维度。

### Rule 2

对应维度满足：

$$
a=b
$$

或者其中一个为：

$$
1
$$

即可兼容。

### Rule 3

Reduction：

`mean(dim=-1)`

通常会删除对应维度。

### Rule 4

使用：

`keepdim=True`

可以把 reduction 后的维度保留成 size=1。

这通常是为了后续 broadcasting。

### Rule 5

Transformer 中常见的 Broadcasting：

- Linear Bias
- RMSNorm
- Attention Mask

因此 broadcasting 不是 PyTorch 的边缘技巧，而是 Transformer 实现中的基本机制。
